# Resource Demand Explanatory

## Problem Framing

**Business question:** Which safehouse patterns most explain rising or falling near-term demand?

**Primary users:** operations leadership, fundraising leadership

**Decision supported:** Support staffing and fundraising planning with a forecast of near-term resident demand by site.

**Modeling goal:** Explanation. This notebook prioritizes interpretable relationships, defensible time ordering, and business understanding over squeezing out marginal predictive lift.

**Target / analysis anchor:** Current regression target: `label_next_active_residents`, forecasting next-month resident load from safehouse-month patterns.

**Submission status:** Paired explanatory analysis with a committed predictive artifact available in the repo for benchmarking and deployment context.

Explanation is the right framing here when leadership needs to understand which patterns consistently move with the outcome and which relationships are too confounded to treat as policy truth. The paired predictive notebook shows how well the signal generalizes on new data; this notebook focuses on what the observed relationships mean and how cautiously they should be used.


## Data Acquisition, Preparation & Exploration

**Data source and grain:** This notebook loads `safehouse_monthly_features`, which contains safehouse-month snapshots that combine occupancy, service, staffing, and operating signals at the site-month grain.

**Reproducible preparation pipeline:** The shared notebook bootstrap and `load_notebook_context(...)` connect this notebook to the same reusable loaders, joins, cleaning rules, and feature derivations used elsewhere in `ml/src`. That keeps the explanatory work tied to the same governed data assets rather than to ad hoc notebook-specific tables.

**Exploration focus:** The code and artifacts for this notebook should be read with attention to segment differences, missingness, potential measurement error, time ordering, and any variable that may be a consequence rather than a cause of the outcome.

**Shared references used by this notebook:**
* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-b-notebook-standardization.md`
* `ml/docs/phase-2-shared-prep.md`
* `ml/docs/phase-4-modeling-framework.md`


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='resource_demand',
    dataset_name='safehouse_monthly_features',
    predictive_impl='resource_demand',
    use_predictive_dataset=False,
)
dataset = context["dataset"]

summarize_frame(dataset)


## Modeling & Feature Selection

This notebook is explanation-first. The preferred model family is whichever gives the clearest defensible story about the relationship between the outcome and the covariates while preserving reasonable empirical support. Transparent regression-style models, grouped comparisons, and interpretable tree summaries are usually more appropriate here than opaque leaderboard-first systems.

Feature selection in this notebook emphasizes:

1. Clear business meaning.
2. Plausible time ordering relative to the outcome.
3. Low risk of direct leakage or post-treatment bias.
4. Stability across subgroups and reporting periods.

The paired predictive artifacts are used only as a benchmark and context for explanation; they do not convert association into causation.


## Evaluation & Interpretation

Evaluation in an explanatory notebook is not just about predictive score. The relationships need to be directionally sensible, stable enough to discuss with stakeholders, and useful for decision-makers who are planning staffing, outreach, case management, or program design.

**Supporting evidence from committed artifacts:** The committed best model is `random_forest_regressor` for target `label_next_active_residents`, selected on `r2` = `1.0000`, using `324` training rows and `117` holdout rows.

**Business interpretation:** A strong explanatory signal means the pattern is consistently associated with the business outcome, so leadership can use it to refine `support staffing and fundraising planning with a forecast of near-term resident demand by site.` without treating the relationship as automatically causal.

**Practical caution:** False negatives matter because they can hide girls, cases, or sites that need early intervention. False positives matter because staff time, bed capacity, and follow-up slots are limited and can be pulled away from higher-need situations.

If the relationships change sharply across time windows, safehouses, donor segments, or channels, leadership should treat the result as context-dependent rather than universal.


In [ ]:
evaluation = load_evaluation_bundle('resource_demand')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


In [ ]:
summarize_frame(holdout_comparison)


## Causal and Relationship Analysis

This is the section where causal language must stay disciplined. Even when a relationship is statistically strong or operationally intuitive, it may still reflect omitted variables, selection effects, measurement design, or reverse causality.

**Most important observable relationships in the committed repo artifacts:** The strongest committed model signals currently surfaced in the repo are `active residents`, `capacity utilization ratio`, `capacity girls`, `capacity gap`, `city General Santos`. These are useful for prioritization and investigation, but they still represent association rather than proof of causation.

**What can be claimed honestly:** These features identify which observed conditions move with the outcome strongly enough to deserve managerial attention.

**What cannot be claimed from this notebook alone:** That changing any one feature will necessarily cause the outcome to improve. Stronger causal claims would require cleaner identification, intervention design, or quasi-experimental evidence beyond the scope of this submission.


## Deployment Notes

**Canonical widgets for the web app:**
* `forecast_widget`
* `insight_summary_card`
* `ranked_table_widget`

**Submission deployment notes:**
* Use a forecast widget and ranked planning table in monthly operations reviews.
* Pair the forecast with recent occupancy context so leaders can see whether the demand signal reflects a sustained load or a temporary shift.

**How this explanatory notebook should surface in the application:**
* use insight summaries, explanation cards, and dashboard annotations before exposing new automated actions
* pair the explanatory narrative with the predictive score when the application needs both ranking and interpretation
* cite the saved repo artifacts when staff ask why a pattern is being highlighted
